In [2]:
# end_to_end_sanity_check.py
"""
Run a small end-to-end test on real data to verify M_inc/M_exc > 0.
"""

import json
import pandas as pd
from config import Config
from trial_graph import PatientClinicalState, TrialStore, compute_matching_indices
from hierarchy import ICD10Hierarchy  # Your fixed class

def sanity_check():
    cfg = Config()
    
    # 1. Load a small sample of patients
    diag_df = pd.read_parquet(f"{cfg.OUTPUT_DIR}/diagnoses_clean.parquet").head(100)
    rx_df = pd.read_parquet(f"{cfg.OUTPUT_DIR}/prescriptions_clean.parquet").head(100)
    labs_df = pd.read_parquet(f"{cfg.OUTPUT_DIR}/labs_clean.parquet").head(100)
    
    patient_states = {
        sid: PatientClinicalState.build_from_tables(sid, diag_df, rx_df, labs_df)
        for sid in diag_df['SUBJECT_ID'].unique()
    }
    
    # 2. Load a small sample of trials
    with open("processed_data/1000_trials/structured_clinical_trials.json", "r") as f:
        trials_data = json.load(f)
    trials_data = trials_data[:10]  # First 10 trials
    
    trial_store = TrialStore.from_records(trials_data)
    
    # 3. Load hierarchy
    hierarchy = ICD10Hierarchy("icd10_hierarchy.csv", log_duplicates=True)
    
    # 4. Compute M_inc/M_exc for all patient-trial pairs
    nonzero_count = 0
    total_count = 0
    
    for pid, state in patient_states.items():
        for tid, trial in trial_store.trials.items():
            total_count += 1
            m_inc, m_exc = compute_matching_indices(state, trial, hierarchy)
            if m_inc > 0 or m_exc > 0:
                nonzero_count += 1
    
    # 5. Report results
    print(f"Total pairs: {total_count}")
    print(f"Nonzero M_inc/M_exc pairs: {nonzero_count}")
    print(f"Fraction: {nonzero_count/total_count*100:.2f}%")
    
    if nonzero_count == 0:
        print("❌ CRITICAL: No matches found. Pipeline will not learn.")
        print("   Possible causes:")
        print("   - Trial codes still don't match patient codes")
        print("   - ICD-10 hierarchy file is empty or misformatted")
        print("   - MedCAT mapping is not producing ICD-10 codes")
    else:
        print("✅ Matches found! Pipeline can proceed.")

if __name__ == "__main__":
    sanity_check()

2026-07-28 13:38:32,394 - INFO - Loaded 10 trials with 118 total criteria
2026-07-28 13:38:32,399 - ERROR - Hierarchy file not found: icd10_hierarchy.csv


FileNotFoundError: [Errno 2] No such file or directory: 'icd10_hierarchy.csv'